# Upload Your Own Image for Processing

Upload any image and it will be automatically denoised and analyzed with object detection

## Setup (Run Once)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import matplotlib.patches as patches
from google.colab import files
from PIL import Image
import io

model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

def detect_objects(image, threshold=0.8):
    img_tensor = F.to_tensor(image).unsqueeze(0).to(device)
    with torch.no_grad():
        predictions = model(img_tensor)[0]
    keep = predictions['scores'] > threshold
    return (
        predictions['boxes'][keep].cpu().numpy(),
        predictions['labels'][keep].cpu().numpy(),
        predictions['scores'][keep].cpu().numpy()
    )

def process_image(image):
    """Process uploaded image: denoise and detect objects"""
    
    # Apply denoising
    gaussian = cv2.GaussianBlur(image, (5, 5), 0)
    bilateral = cv2.bilateralFilter(image, 9, 75, 75)
    nlm = cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)
    
    # Calculate PSNR
    psnr_gaussian = peak_signal_noise_ratio(image, gaussian)
    psnr_bilateral = peak_signal_noise_ratio(image, bilateral)
    psnr_nlm = peak_signal_noise_ratio(image, nlm)
    
    # Detect objects
    boxes_orig, labels_orig, scores_orig = detect_objects(image, threshold=0.8)
    boxes_nlm, labels_nlm, scores_nlm = detect_objects(nlm, threshold=0.8)
    
    # Display results
    print("PSNR Results:")
    print(f"  Gaussian: {psnr_gaussian:.2f} dB")
    print(f"  Bilateral: {psnr_bilateral:.2f} dB")
    print(f"  NLM: {psnr_nlm:.2f} dB")
    print(f"\nDetections: {len(boxes_orig)} objects")
    print("Detected objects:")
    for label, score in zip(labels_orig, scores_orig):
        print(f"  - {COCO_CLASSES[label]}: {score:.2f}")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    for ax, img, boxes, labels, scores, title, color in [
        (axes[0], image, boxes_orig, labels_orig, scores_orig, 'Original Image (threshold=0.8)', 'red'),
        (axes[1], nlm, boxes_nlm, labels_nlm, scores_nlm, 'Denoised Image (threshold=0.8)', 'green')
    ]:
        ax.imshow(img)
        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f'{COCO_CLASSES[label]}: {score:.2f}', 
                   bbox=dict(facecolor=color, alpha=0.5), fontsize=10, color='white')
        ax.set_title(title, fontsize=14)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\nKEY FINDINGS FOR YOUR REPORT:")
    print(f"1. Image quality improved by {psnr_nlm - psnr_gaussian:.2f} dB")
    print(f"2. Original: {len(boxes_orig)} detections, Denoised: {len(boxes_nlm)} detections")
    if len(scores_orig) > 0 and len(scores_nlm) > 0:
        print(f"3. Confidence change: {(scores_nlm.mean() - scores_orig.mean())*100:.1f}%")

print("Setup complete. Ready to process images.")

## Upload and Process Your Image

In [ ]:
print("Click 'Choose Files' and select your image...")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nProcessing {filename}...")
    
    # Load image
    img = Image.open(io.BytesIO(uploaded[filename]))
    img = img.convert('RGB')
    img_array = np.array(img)
    
    # Process
    process_image(img_array)

## Process Multiple Images

In [ ]:
print("Upload multiple images...")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*70}")
    print(f"Processing: {filename}")
    print('='*70)
    
    img = Image.open(io.BytesIO(uploaded[filename]))
    img = img.convert('RGB')
    img_array = np.array(img)
    
    process_image(img_array)